# Process All Videos — Colab T4 GPU

Colab T4 has sm_75 which works with default PyTorch.

**Setup:** Runtime → Change runtime type → **T4 GPU**

**Data:** Mounts Google Drive directly (no dataset upload needed)

In [ ]:
# Cell 1: Setup + Mount Drive
import os, sys, json, subprocess, warnings, shutil
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# Install deps
subprocess.run([sys.executable, '-m', 'pip', 'install', 'yt-dlp', '-q'], capture_output=True)

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import librosa
from tqdm import tqdm
from transformers import AutoModel

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if str(device) == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    # Verify group_norm works (the op that failed on P100)
    gn = torch.nn.GroupNorm(1, 64).to(device)
    _ = gn(torch.randn(1, 64, 100).to(device))
    print('✅ GroupNorm works on GPU')
else:
    print('⚠️ No GPU! Enable: Runtime → Change runtime type → T4 GPU')

In [ ]:
# Cell 2: Mount Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

BASE = '/content/drive/MyDrive/standup4ai'
WORK = '/content/process255'
os.makedirs(WORK, exist_ok=True)
os.makedirs(f'{WORK}/features', exist_ok=True)
os.makedirs(f'{WORK}/audio', exist_ok=True)

# Find labels
LABEL_DIR = None
for root, dirs, files in os.walk(BASE):
    csvs = [f for f in files if f.endswith('.csv')]
    if len(csvs) > 50:
        LABEL_DIR = root
        break

if not LABEL_DIR:
    LABEL_DIR = f'{BASE}/seq-Standup4AI/dataset/en_uk'

print(f'BASE: {BASE}')
print(f'WORK: {WORK}')
print(f'LABEL_DIR: {LABEL_DIR}')

# Count labels recursively
total_labels = 0
for root, dirs, files in os.walk(LABEL_DIR):
    total_labels += len([f for f in files if f.endswith('.csv')])
print(f'Total label CSVs: {total_labels}')

In [ ]:
# Cell 3: Video IDs + Load WavLM on GPU
# First 50 videos for this run
ALL_VIDEOS = [
    '-UPIA46hBZs','-vcKXr6WBNc','0AvUvJ_S2Os','0Pl51hxcK-o','0g7nezWZyfY',
    '0zpUnJSG0EQ','18H1aeoGybw','18rLwnvxOU0','1ILQmgHvtd4','1Uo27tH3JQ4',
    '1pPnJut3KLw','1tO9MWWOgHk','1u-pq9LLWlU','21gOjz-Xk7s','2SUfHIbT0HI',
    '2axWotdMFsw','2ql8QJWmNM8','3TgRGK1vrzs','3ZTClwMxpmM','3new05S61w4',
    '41piF6uPhXg','482LeT9UT7I','4ZiXvhSxnD4','53JXuJGmhoU','5bKcTy3zag4',
    '5cdoHY0ziVA','5gp79fSWHy0','66CyaeFWucM','6Ofc2A75zuw','76r8IcowEsE',
    '7E7la6BCpRc','7Gw1NjZ13fA','7VkAFkK3bwQ','7cBFWZDXlHA','7gRo0nF1yS0',
    '7kULz2NevT4','8CoHAczz9pY','8EUpV_qyEpc','8eYSNXOsyoo','8nltoWdciws',
    '90s9HfZhM0Y','9DwiBEVDdUE','9h7-OMYItDI','9yPco6WNYG0','AES4jzE513Y',
    'AEnlxaPVtK8','AI69HZWZ26c','A_EIL1ojfK4','Azl5GJuYqE0','B9jLEExvazc'
]
print(f'This batch: {len(ALL_VIDEOS)} videos')

# Load WavLM on GPU (should work on T4/sm_75)
SR_WAVLM, SR_PROSODY = 16000, 22050
wavlm = AutoModel.from_pretrained('microsoft/wavlm-base')
wavlm.to(device); wavlm.eval()
print('✓ WavLM loaded on GPU')

In [ ]:
# Cell 4: Define Feature Extractors
def prosody23(y, sr):
    f = []
    try:
        f0, vd, _ = librosa.pyin(y, fmin=50, fmax=500, sr=sr)
        f0c = f0[~np.isnan(f0)]; v = vd[~np.isnan(f0)]
        f.extend([np.mean(f0c) if len(f0c)>0 else 0,
                  np.std(f0c) if len(f0c)>0 else 0,
                  np.max(f0c) if len(f0c)>0 else 0,
                  np.min(f0c) if len(f0c)>0 else 0,
                  np.mean(v) if len(v)>0 else 0])
    except: f.extend([0]*5)
    hop = 512
    rms = librosa.feature.rms(y=y, hop_length=hop)[0]
    f.extend([np.mean(rms), np.std(rms), np.max(rms), np.min(rms), np.max(rms)-np.min(rms)])
    dur = len(y)/sr
    f.extend([dur, dur/(np.sum(rms>np.mean(rms))+1)])
    try:
        sc = librosa.feature.spectral_centroid(y=y, sr=sr, hop_length=hop)[0]
        sb = librosa.feature.spectral_bandwidth(y=y, sr=sr, hop_length=hop)[0]
        sf2 = librosa.feature.spectral_flatness(y=y, hop_length=hop)[0]
        zcr = librosa.feature.zero_crossing_rate(y, hop_length=hop)[0]
        f.extend([np.mean(sc), np.mean(sb), np.mean(sf2), np.mean(zcr), np.std(zcr)])
    except: f.extend([0]*5)
    try:
        yh, _ = librosa.effects.hpss(y)
        hnr = np.mean(np.abs(yh))/(np.mean(np.abs(y))+1e-8)
        f.extend([hnr, np.mean(np.abs(y)), np.std(y), np.max(np.abs(y)), 0, 0])
    except: f.extend([0]*6)
    return np.array(f[:23], dtype=np.float32)

def word_features(y16, y22, t0, t1):
    dur = t1 - t0
    if dur < 0.005: return None
    s16, e16 = int(t0*SR_WAVLM), min(int(t1*SR_WAVLM), len(y16))
    c16 = y16[s16:e16]
    if len(c16) < int(0.01*SR_WAVLM): return None
    target_len = 5 * SR_WAVLM
    if len(c16) < target_len:
        c16 = np.pad(c16, (0, target_len-len(c16)))
    else:
        c16 = c16[:target_len]
    with torch.no_grad():
        inp = torch.tensor(c16/32768.0, dtype=torch.float32).unsqueeze(0).to(device)
        wemb = wavlm(inp).last_hidden_state.mean(dim=1).squeeze().cpu().numpy()
    s22, e22 = int(t0*SR_PROSODY), min(int(t1*SR_PROSODY), len(y22))
    pros = prosody23(y22[s22:e22], SR_PROSODY)
    return np.concatenate([wemb, pros])

print('✅ Feature extractor ready')

In [ ]:
# Cell 5: Process Videos (Download Audio + Extract Features)
def get_audio(vid):
    """Get audio from Drive or YouTube."""
    # Try Drive first
    for folder in [f'{BASE}/audio', f'{BASE}/audio_1000']:
        p = f'{folder}/{vid}.m4a'
        if os.path.exists(p):
            dst = f'{WORK}/audio/{vid}.m4a'
            if not os.path.exists(dst):
                shutil.copy(p, dst)
            return dst
    
    # Check working dir
    for ext in ['.wav', '.m4a']:
        p = f'{WORK}/audio/{vid}{ext}'
        if os.path.exists(p): return p
    
    # Fallback: yt-dlp
    base = f'{WORK}/audio/{vid}'
    cmd = ['yt-dlp', '-f', 'bestaudio',
           '-o', f'{base}.%(ext)s',
           f'https://www.youtube.com/watch?v={vid}',
           '--no-playlist', '--quiet', '--socket-timeout', '90',
           '--extract-audio', '--audio-format', 'wav']
    try:
        subprocess.run(cmd, capture_output=True, text=True, timeout=180)
        for ext in ['.wav', '.m4a']:
            if os.path.exists(f'{base}{ext}'): return f'{base}{ext}'
    except: pass
    return None

DONE_FILE = f'{WORK}/features_done.json'
done = set()
if os.path.exists(DONE_FILE):
    with open(DONE_FILE) as f: done = set(json.load(f))
print(f'Resuming: {len(done)} done')

# Recursively find label file
def find_label(vid):
    for root, dirs, files in os.walk(LABEL_DIR):
        if f'{vid}.csv' in files:
            return os.path.join(root, f'{vid}.csv')
    return None

processed, errors, no_audio = 0, [], []
for vid in tqdm(ALL_VIDEOS, desc='Processing'):
    if vid in done: continue
    
    ap = get_audio(vid)
    if not ap:
        no_audio.append(vid)
        continue
    
    lp = find_label(vid)
    if not lp: continue
    
    try:
        y22, _ = librosa.load(ap, sr=SR_PROSODY, mono=True)
        y16, _ = librosa.load(ap, sr=SR_WAVLM, mono=True)
        df = pd.read_csv(lp)
        
        feats, lbls = [], []
        for _, row in df.iterrows():
            try:
                ts = eval(str(row['timestamp']))
                t0, t1 = float(ts[0]), float(ts[1])
                feat = word_features(y16, y22, t0, t1)
                lbl = str(row.get('label', 'O')).strip()
                if feat is not None:
                    feats.append(feat)
                    lbls.append(1 if lbl in ('B','I','L') else 0)
            except: pass
        
        if feats:
            np.save(f'{WORK}/features/{vid}_features.npy', np.array(feats, dtype=np.float32))
            np.save(f'{WORK}/features/{vid}_labels.npy', np.array(lbls, dtype=np.int32))
            done.add(vid)
            processed += 1
            pos_rate = sum(lbls)/max(len(lbls),1)
            print(f'  ✓ {vid}: {len(feats)} words ({100*pos_rate:.1f}% laugh)')
        
        del y16, y22
        import gc; gc.collect()
        torch.cuda.empty_cache() if str(device)=='cuda' else None
        
        if processed % 10 == 0:
            with open(DONE_FILE, 'w') as f:
                json.dump(sorted(done), f)
    except Exception as e:
        errors.append((vid, str(e)[:200]))

with open(DONE_FILE, 'w') as f:
    json.dump(sorted(done), f)

print(f'\n=== COMPLETE ===')
print(f'Processed this run: {processed}/{len(ALL_VIDEOS)}')
print(f'Total done: {len(done)}')
print(f'No audio: {len(no_audio)}')
print(f'Errors: {len(errors)}')